# Chapter 03C - Carry a note instead of cutting off the clue

Our seats kept the words in order in [03B](03B-counts-and-fixed-window-language-models.ipynb).
They still left `red` and `blue` outside the window. What if each word could pass a note
to the next one? The final prediction could then use something learned from the opening color.

Keep the same sentences, answers, and training/test split. First try a simple RNN and watch
its notes change. Then try an LSTM: it adds learned controls over what to keep and rewrite.
Chapter 01 explains those controls; here we put them to work on words.

We'll also use 03A's inherited training loop and check that the loss really changes the
embedding rows as well as the recurrent layers and word-scoring layer.

**Code-only view:** [Python companion](03C-fixed-windows-to-recurrent-language-models.py).

# Fixed windows to running notes

**Guiding question:** Can a running note use the color that the three-word window could not see?

You ended [03B](03B-counts-and-fixed-window-language-models.ipynb) with identical windows
and different required answers. Keep the same eight sentences, six training rows, and two held-out
rows. The only new route is a note passed from one word to the next.

| Attempt | Complaint | Evidence |
|---|---|---|
| Rebuild the window | Color is outside | Final-token accuracy and equality |
| Carry a running note | A route is not a guarantee | Accuracy and swapped-clue probabilities |
| Inspect the note | It may forget later | Recorded state trace and animation |

Remember the [sequence-memory lesson](../01-sequence-memory/README.md)? Notes can lose useful
details as we keep rewriting them. A **vanilla recurrent neural network (RNN)** is the simplest
version: read a word and the old note, then write the next note using the same learned update.
Later we'll use LSTM's gates, the learned controls over what to keep and rewrite.
No shared state or downloads. [Handwritten companion](03C-fixed-windows-to-recurrent-language-models-theory.md).

## 1. Keep the old failure in view

**Predict:** Will refitting the same three-word input remove its blind spot?

![A recurrent reader passes a note between words. The note can carry the clue beyond a window but can also forget.](images/03-evo-03-fading-notecard.png)
*A running note has a route beyond the window, but recall is not guaranteed.*


In [ ]:
import random
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
random.seed(17); np.random.seed(17); torch.manual_seed(17)
torch.set_num_threads(1)
started = time.perf_counter()

In [ ]:
corpus = [f"{clue} {first} {second} now {answer}".split()
          for clue, answer in [("red", "warm"), ("blue", "cool")]
          for first in ["wait", "rest"] for second in ["quietly", "calmly"]]
train_rows = [r for r in corpus if r[1:3] != ["rest", "calmly"]]
test_rows = [r for r in corpus if r[1:3] == ["rest", "calmly"]]
for row in corpus:
    print(" ".join(row), "|", "held out" if row in test_rows else "training")

IDs still mean addresses. Each retrieves a trainable row, or **embedding**, before either network
uses it. The window joins three rows in order. The running reader updates one note after each row.
Neither receives the answer as part of the input it is graded on.

In [ ]:
vocab = ["<start>"] + sorted({word for row in corpus for word in row})
to_id = {word: i for i, word in enumerate(vocab)}
def encode(rows):
    return torch.tensor([[to_id[word] for word in row] for row in rows])
train_ids, test_ids = encode(train_rows), encode(test_rows)
print("ID addresses:", to_id)

In [ ]:
WINDOW = 3
def window_pairs(rows):
    contexts, answers = [], []
    for row in rows:
        for position in range(1, len(row)):
            prefix = ["<start>"] * WINDOW + row[:position]
            contexts.append([to_id[word] for word in prefix[-WINDOW:]])
            answers.append(to_id[row[position]])
    return torch.tensor(contexts), torch.tensor(answers)
class WindowLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.rows = nn.Embedding(len(vocab), 8)
        self.predictor = nn.Sequential(nn.Linear(WINDOW * 8, 24), nn.Tanh(),
                                       nn.Linear(24, len(vocab)))
    def forward(self, context):
        return self.predictor(self.rows(context).flatten(1))

In [ ]:
window_model = WindowLM()
x, y = window_pairs(train_rows)
optimizer = torch.optim.Adam(window_model.parameters(), lr=0.03)
for step in range(300):
    loss = F.cross_entropy(window_model(x), y)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
with torch.no_grad():
    window_probs = window_model(test_ids[:, -4:-1]).softmax(-1)
    window_accuracy = (window_probs.argmax(-1) == test_ids[:, -1]).float().mean().item()
print(f"Fixed-window final-token accuracy: {window_accuracy:.1%}")
print("Takeaway: fitting again does not give the window a missing color.")

**Your turn:** Change only `inspect_row`. Does selecting the other clue change the visible window?

In [ ]:
inspect_row = 0  # CHANGE THIS: 1 selects the other clue
print("Whole prefix:", test_rows[inspect_row][:-1])
print("Window input:", test_rows[inspect_row][-4:-1])

**More training didn't bring the color back.** Let's try carrying it forward in a note.

## 2. Read a word, update the note, predict the next word

**Predict:** After `red rest calmly now`, can a note still distinguish the first word from `blue`?

```text
red  -> note -> rest -> note -> calmly -> note -> now -> note -> warm?
blue -> note -> rest -> note -> calmly -> note -> now -> note -> cool?
```

The note is a **hidden state**: a vector the network updates as it reads. Unlike an embedding
row stored for a word ID, this note depends on the words read so far.
The model returns a note at every position. An output layer turns each note into next-token scores.
Input `red wait quietly now` is graded against `wait quietly now warm`. The answer is one position
ahead. This shift prevents the current answer from being its own input.
Both models practice on the same next-word pairs, but they have different numbers of weights.
Even equal practice time wouldn't make this a test that changes only the architecture.

In [ ]:
class RunningLM(nn.Module):
    def __init__(self, width=24):
        super().__init__()
        self.rows = nn.Embedding(len(vocab), 8)
        self.reader = nn.RNN(8, width, batch_first=True)
        self.head = nn.Linear(width, len(vocab))
    def forward(self, tokens):
        notes, last = self.reader(self.rows(tokens))
        return self.head(notes), notes

torch.manual_seed(17)
running_model = RunningLM()
optimizer = torch.optim.Adam(running_model.parameters(), lr=0.02)
losses = []
for step in range(350):
    scores, _ = running_model(train_ids[:, :-1])
    loss = F.cross_entropy(scores.flatten(0, 1), train_ids[:, 1:].flatten())
    optimizer.zero_grad(); loss.backward()
    nn.utils.clip_grad_norm_(running_model.parameters(), 1.0)
    optimizer.step()
    losses.append(loss.item())
with torch.no_grad():
    scores, notes = running_model(test_ids[:, :-1])
    recurrent_probs = scores[:, -1].softmax(-1)
    recurrent_accuracy = (recurrent_probs.argmax(-1) == test_ids[:, -1]).float().mean().item()
    clue_effect = (recurrent_probs[0] - recurrent_probs[1]).abs().max().item()
print(f"RNN training loss: {losses[0]:.4f} -> {losses[-1]:.4f}")
print(f"Held-out final-token accuracy: window {window_accuracy:.1%}; RNN {recurrent_accuracy:.1%}")
print(f"Largest final probability change after swapping only the clue: {clue_effect:.6f}")
for name, model in [("window", window_model), ("RNN", running_model)]:
    print(name, "parameters:", sum(p.numel() for p in model.parameters()))
if recurrent_accuracy > window_accuracy:
    print("Takeaway: this run used the clue beyond the fixed window on the tiny held-out pair.")
else:
    print("Takeaway: the note has a route, but this run did not establish an accuracy gain.")

**Follow one sentence:** fetch its word rows, then let the RNN update its note four times.
After each word, the output layer guesses the following word. Working backward through that
loss teaches all three parts: the rows, the recurrent update, and the output layer.
Gradient clipping caps the feedback's size before the optimizer step. It can stop an oversized
update; it can't recover a detail the note has forgotten.

**Your turn:** Change only `clue_word`. Keep the filler words. Inspect the final choice.

In [ ]:
clue_word = "red"  # CHANGE THIS: "blue"
probe = torch.tensor([[to_id[w] for w in [clue_word, "rest", "calmly", "now"]]])
with torch.no_grad():
    final = running_model(probe)[0][0, -1].softmax(-1)
print("warm:", round(final[to_id["warm"]].item(), 4),
      "cool:", round(final[to_id["cool"]].item(), 4))

**That worked on a short task. What did the note actually keep?**

## 3. Watch the two notes separate

**Predict:** After reading the same filler words, must the red and blue notes become identical?
The curve measures how far apart their hidden-state vectors are at each position.
The animation replays those same measurements; it doesn't invent a fading effect.
The static plot lets you read the evidence without playing the animation.

Keep the answer accuracy beside this picture. Two notes can differ without that difference
helping the model choose the correct word.

In [ ]:
distances = (notes[0] - notes[1]).norm(dim=-1).numpy()
labels = ["red / blue", "rest", "calmly", "now"]
fig, ax = plt.subplots(figsize=(6, 2.5))
ax.plot(labels, distances, marker="o", color="#ef8a23")
ax.set(ylabel="Distance between learned notes", title="Same later words, different opening clue")
plt.show()
print("Recorded distances:", distances.round(4).tolist())
print("Takeaway: read state differences alongside behavior; visible difference alone is not recall.")

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
fig, ax = plt.subplots(figsize=(6, 2.5))
ax.set(xlim=(-0.2, 3.2), ylim=(0, max(distances) * 1.15),
       ylabel="Distance between notes", title="Watch the measured note travel")
ax.set_xticks(range(4), labels)
line, = ax.plot([], [], "o-", color="#ef8a23")
def frame(position):
    line.set_data(range(position + 1), distances[:position + 1])
    return (line,)
animation = FuncAnimation(fig, frame, frames=4, interval=600, blit=True)
plt.close(fig)
display(HTML(animation.to_jshtml()))

**Your turn:** Change only `position_to_read`. Which position has the biggest recorded difference?

In [ ]:
position_to_read = 3  # CHANGE THIS: 0, 1, or 2
print(labels[position_to_read], float(distances[position_to_read]))

**Four words went well. Hundreds are another question.**

## 4. What the note fixes, and what it does not

A **long short-term memory network (LSTM)** adds controls over what to keep and rewrite,
with a more protected path for carrying memory. The
[Chapter 01 gate experiments](../01-sequence-memory/01A-sequence-memory-theory.ipynb) show
why that can help, not why every LSTM must beat every RNN. A **gated recurrent unit (GRU)**
is another running-note design with learned controls. We'll use one in 03D; you won't need
a new set of gate formulas. Neither design automatically lets a writer reopen earlier notes.

| Teaching mechanism | Production counterpart | What changes | What stays |
|---|---|---|---|
| Four note updates | Recurrent text processing | Longer texts and larger states | Same update repeats in order |
| Gradient cap | Gradient clipping | Threshold and optimizer | Cap is not a memory repair |
| Held-out filler combination | Evaluation split | More examples and seeds | Do not grade only memorized rows |

### Check the results

In [ ]:
print({"window_final_accuracy": window_accuracy, "RNN_final_accuracy": recurrent_accuracy,
       "swapped_clue_probability_effect": clue_effect,
       "RNN_parameters": sum(p.numel() for p in running_model.parameters()),
       "window_parameters": sum(p.numel() for p in window_model.parameters())})
print(f"Elapsed CPU wall time: {time.perf_counter() - started:.1f}s")

### What we tried, explained, and left for later

- **What we built and checked:** same corpus and split, ordered-window baseline, vanilla RNN training,
  shifted targets, parameter counts, clue intervention, state-distance storyboard and animation.
- **What we explained:** hidden state, clipping, short-task limits, LSTM's memory-route
  purpose, and why state difference is not sufficient behavioral evidence.
- **What we're leaving for later:** gate internals and fading teaching signals (owned by
  [sequence memory](../01-sequence-memory/README.md)); GRU internals; long-context benchmarks;
  bidirectional next-token reading (future answers are not legal evidence).

**Watch out for these traps:** avoid future-word leakage; do not call two test sentences a benchmark;
do not equate nonzero state distance with correct use of a clue.

**The question we're left with:** You can continue one stream. What if the input and answer are different
sequences? Revisit the [03A reader/writer preview](03A-training-skeleton-and-word-objectives.ipynb),
then [03D](03D-recurrent-encoder-decoder-to-attention.ipynb).

## 5. Keep the practice routine; try LSTM memory

The RNN carried the clue beyond the window. Now let's use the LSTM memory we met in Chapter 01,
without writing another training framework.

Think of LSTM as carrying both a working note and a more protected memory line, with learned
controls over what to keep and rewrite. In PyTorch those are the hidden and cell states.
They are vectors of numbers, not literal text or guaranteed perfect memories.

**Predict:** If we change the running-note mechanism to `nn.LSTM`, do the word lookup,
next-word answers, or optimizer loop need to change?

```text
input IDs [B, L] -> fetch rows [B, L, 8]
-> RNN or LSTM notes [B, L, 24] -> word scores [B, L, V]
correct answers [B, L]: the word following each input
```

The output layer reads the hidden output at each position. The cell state helps carry memory;
it is not a second answer to grade. We still compare one set of word scores with one correct
next-word ID at each position.

This is still recurrence, not a Transformer. The model updates its notes one position at a time.
Carrying a better note is also not the same as being able to reopen every earlier source position.

In [ ]:
from language_model import TrainableLanguageModel
from torch.utils.data import DataLoader, TensorDataset


class RecurrentLanguageModel(TrainableLanguageModel):
    def __init__(self, vocabulary_size, cell_type="lstm", embedding_width=8, hidden_width=24):
        super().__init__()
        if cell_type not in {"rnn", "lstm"}:
            raise ValueError("cell_type must be 'rnn' or 'lstm'")
        self.embedding = nn.Embedding(vocabulary_size, embedding_width)
        recurrent_layer = nn.LSTM if cell_type == "lstm" else nn.RNN
        self.reader = recurrent_layer(embedding_width, hidden_width, batch_first=True)
        self.output = nn.Linear(hidden_width, vocabulary_size)

    def forward(self, input_ids, target_ids=None):
        notes, _ = self.reader(self.embedding(input_ids))
        return self.output(notes)

    def generate(self, prefix_ids, steps):
        self.eval()
        generated = prefix_ids.clone()
        with torch.inference_mode():
            for _ in range(steps):
                next_id = self(generated)[:, -1].argmax(dim=-1, keepdim=True)
                generated = torch.cat([generated, next_id], dim=1)
        return generated

### Where does the LSTM use our shared loop?

Look at `lstm_model.fit_language_model(...)` below. `RecurrentLanguageModel` doesn't define
that method; it inherits it from `TrainableLanguageModel`.
The shared loop calls `training_loss`, which asks this model's `forward` to make the guesses.

We already paired `train_ids[:, :-1]` with `train_ids[:, 1:]`: each word is graded against
the following word. No extra target shift is needed inside this model.
03A's encoder-decoder did that shift inside the model instead.

Leave the earlier RNN experiment and its clipped gradients alone. This LSTM run shows that
the same training routine works here; its different setup is not a fair contest to decide
which recurrent layer is better.

In [ ]:
torch.manual_seed(17)
lstm_model = RecurrentLanguageModel(len(vocab))
lstm_inputs, lstm_targets = train_ids[:, :-1], train_ids[:, 1:]
lstm_loader = DataLoader(TensorDataset(lstm_inputs, lstm_targets), batch_size=len(train_ids))
lstm_optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.03)
embedding_before = lstm_model.embedding.weight.detach().clone()

assert RecurrentLanguageModel.fit_language_model is TrainableLanguageModel.fit_language_model
lstm_losses = lstm_model.fit_language_model(
    lstm_loader, nn.CrossEntropyLoss(), lstm_optimizer, epochs=350,
)
lstm_model.eval()
with torch.inference_mode():
    lstm_logits = lstm_model(test_ids[:, :-1])
    lstm_accuracy = (lstm_logits[:, -1].argmax(-1) == test_ids[:, -1]).float().mean().item()
    _, (hidden, cell) = lstm_model.reader(lstm_model.embedding(test_ids[:, :-1]))
embedding_change = (lstm_model.embedding.weight.detach() - embedding_before).abs().max().item()

print(f"LSTM loss: {lstm_losses[0]:.4f} -> {lstm_losses[-1]:.4f}")
print(f"LSTM held-out final-token accuracy: {lstm_accuracy:.1%}")
print("Hidden / cell shapes:", tuple(hidden.shape), tuple(cell.shape))
print(f"Largest learned embedding change: {embedding_change:.6f}")
assert lstm_logits.shape == (len(test_ids), test_ids.shape[1] - 1, len(vocab))
assert lstm_losses[-1] < lstm_losses[0]
assert embedding_change > 0

### Now take away the answer key

**Predict:** If you change the final input word, should an earlier prediction change?
It shouldn't: that earlier step hasn't read the final word yet. We'll compare the earlier
scores directly, rather than trusting a sentence that happens to look sensible.

For generation, feed the model's last guess back in and ask again. This helper rereads the
whole prefix each time so the code stays easy to follow. It doesn't cache state, hasn't
learned an end marker, and stops after a fixed number of steps. Simple to inspect, not a
fast generation engine. A good result on these few sentences is still only a tiny-task result.

In [ ]:
prefix = test_ids[:1, :-1]
changed_prefix = prefix.clone()
changed_prefix[:, -1] = to_id["wait"]
with torch.inference_mode():
    original_scores = lstm_model(prefix)
    changed_scores = lstm_model(changed_prefix)
assert torch.allclose(original_scores[:, :-1], changed_scores[:, :-1])
continuation = lstm_model.generate(prefix, steps=1)
print("Generated continuation:", [vocab[index] for index in continuation[0].tolist()])
print("Takeaway: embeddings and LSTM weights learned together; earlier logits do not read future tokens.")

### What we checked - and where to go next

- **What we built and checked:** inherited LSTM training, shifted sequence labels, embedding-weight
  updates, hidden/cell shapes, a future-token intervention, and bounded greedy continuation.
- **Explained:** RNN/LSTM selection changes the recurrent update, not the optimization contract.
- **Out of scope:** a controlled long-delay LSTM-versus-RNN benchmark, state-cached generation,
  and general language quality. Chapter 01 owns the controlled memory experiments.

**But what about this?** continuing one stream does not define how a completed source conditions
a separate answer. Revisit 03A's small encoder-decoder preview if needed, then continue to
[03D](03D-recurrent-encoder-decoder-to-attention.ipynb), where source access is the measured variable.